# Ensemble Foundations — Voting, Averaging, Bagging, and Stacking

## 1. What is an ensemble?
An ensemble combines predictions from several models. The motivation is statistical: models make different errors, and averaging partially independent errors can reduce variance. An ensemble helps most when its members are individually useful and make sufficiently different mistakes.

## 2. Main patterns
- **Voting / averaging:** combine predictions from models trained on the same data.
- **Bagging:** train models on bootstrap resamples and aggregate their predictions. Bootstrap samples are drawn with replacement.
- **Random Forest:** bagging of trees plus random feature subsets at split time.
- **Boosting:** fit models sequentially, with later models correcting previous errors.
- **Stacking:** train a meta-model on out-of-fold predictions from base learners.

For regression, average numeric predictions. For classification, hard voting combines labels; soft voting averages class probabilities.

## 3. Why averaging can help

If each model's prediction error has variance $\sigma^2$ and pairwise error correlation $\rho$, the variance of the mean of M exchangeable errors is

$$
\mathrm{Var}(\bar e)=\sigma^2\left(\rho+\frac{1-\rho}{M}\right).
$$

When errors are uncorrelated ($\rho=0$), variance falls as $1/M$. When errors are perfectly correlated ($\rho=1$), averaging does not help. This equation is an intuition under a simplified equal-variance/equal-correlation model, not a universal guarantee.

For bootstrap aggregation, a common heuristic is that unstable, high-variance learners such as deep trees benefit more than stable learners such as ordinary least squares.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import BaggingRegressor, VotingRegressor, StackingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

SEED = 42
X, y = make_regression(n_samples=900, n_features=12, n_informative=7, noise=18, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)
print("Train shape:", X_train.shape)


Train shape: (675, 12)


## 4. Bootstrap samples by hand

A bootstrap sample has the same number of rows as the original training data, but samples row indices with replacement. Some rows appear multiple times and some are omitted. For a bootstrap sample of size n, the expected fraction of unique original observations is approximately 1 - e⁻¹ ≈ 63.2%.

In [2]:
rng = np.random.default_rng(SEED)
n = len(X_train)
bootstrap_indices = rng.choice(n, size=n, replace=True)
print("Rows sampled:", len(bootstrap_indices))
print("Unique rows:", len(np.unique(bootstrap_indices)))
print("Unique-row fraction:", round(len(np.unique(bootstrap_indices))/n, 3))
assert len(bootstrap_indices) == n


Rows sampled: 675
Unique rows: 416
Unique-row fraction: 0.616


## 5. Bagging from scratch (ensemble mechanics)

This is an ensemble implementation from scratch around a library base learner: each member is a decision tree trained on its own bootstrap sample, and predictions are averaged. The tree learner itself was covered in the decision-tree notebook.

The ensemble reduces variance when trees' errors are not perfectly correlated. It does not fix systematic bias automatically.

In [3]:
def fit_bagging_regressor(X, y, n_estimators=40, max_depth=None, random_state=42):
    X = np.asarray(X); y = np.asarray(y)
    rng = np.random.default_rng(random_state)
    models = []
    n = len(X)
    for _ in range(n_estimators):
        idx = rng.choice(n, size=n, replace=True)
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=int(rng.integers(0, 2**31-1)))
        tree.fit(X[idx], y[idx])
        models.append(tree)
    return models

def predict_bagging_regressor(models, X):
    return np.mean([model.predict(X) for model in models], axis=0)

scratch_models = fit_bagging_regressor(X_train, y_train, n_estimators=40, max_depth=None, random_state=SEED)
pred_scratch = predict_bagging_regressor(scratch_models, X_test)
print("Scratch bagging MSE:", mean_squared_error(y_test, pred_scratch))
print("Scratch bagging R²:", r2_score(y_test, pred_scratch))


Scratch bagging MSE: 5192.995366793313
Scratch bagging R²: 0.8184231483913058


## 6. Compare with scikit-learn

Library results may not match exactly because bootstrap indices, random-number generators, tree randomness, and parallel scheduling can differ. We compare held-out metrics rather than requiring identical predictions.

In [4]:
lib = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=40, max_samples=1.0, bootstrap=True,
    random_state=SEED, n_jobs=-1
).fit(X_train, y_train)
pred_lib = lib.predict(X_test)
print("sklearn bagging MSE:", mean_squared_error(y_test, pred_lib))
print("sklearn bagging R²:", r2_score(y_test, pred_lib))
assert np.isfinite(pred_scratch).all() and np.isfinite(pred_lib).all()


Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/spreadsheet_warmup.py", line 785, in warm_spreadsheet_runtime
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/spreadsheet_warmup.py", line 718, in _warm_feature_flows
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/spreadsheet_warmup.py", line 509, in _warm_drawing_and_image_flows
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/generated/interface/models.py", line 32861, in bring_to_front
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/rpc/remote.py", line 764, in __call__
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/rpc/client.py", line 175, in call
artifact_tool.rpc.client.Remot

Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/spreadsheet_warmup.py", line 785, in warm_spreadsheet_runtime
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/spreadsheet_warmup.py", line 718, in _warm_feature_flows
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/spreadsheet_warmup.py", line 509, in _warm_drawing_and_image_flows
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/generated/interface/models.py", line 32861, in bring_to_front
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/rpc/remote.py", line 764, in __call__
  File "/tmp/tmp.QlpUkOUI9D/artifact_tool_v2-2.8.66/artifact_tool/rpc/client.py", line 175, in call
artifact_tool.rpc.client.Remot

sklearn bagging MSE: 4880.397911661299
sklearn bagging R²: 0.8293533452651017


## 7. Voting and stacking

Voting combines separately trained models. Stacking learns how to combine models; its meta-model must be trained on out-of-fold predictions, not in-sample predictions from base learners, or leakage can make evaluation overly optimistic.

In [5]:
voting = VotingRegressor([
    ("linear", LinearRegression()),
    ("ridge", Ridge(alpha=2.0)),
    ("knn", KNeighborsRegressor(n_neighbors=15))
]).fit(X_train, y_train)
stacking = StackingRegressor(
    estimators=[
        ("tree", DecisionTreeRegressor(max_depth=5, random_state=SEED)),
        ("ridge", Ridge(alpha=2.0)),
        ("knn", KNeighborsRegressor(n_neighbors=15))
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5, n_jobs=-1
).fit(X_train, y_train)
print("Voting R²:", r2_score(y_test, voting.predict(X_test)))
print("Stacking R²:", r2_score(y_test, stacking.predict(X_test)))


Voting R²: 0.9602957758762027
Stacking R²: 0.9887032308217759


## 8. Hyperparameters and best practices

- `n_estimators`: more members usually stabilize results but increase training and prediction cost.
- `max_samples`: controls bootstrap sample size.
- `bootstrap`: sample with replacement for classic bagging; sampling without replacement is a different variant.
- `max_features`: can decorrelate members by limiting feature choices.
- Use held-out or cross-validated evaluation; do not tune on the final test set.
- Bagging is most useful for unstable learners; if base errors are strongly correlated, diversify the learners or features.
- In stacking, use cross-validation for meta-features and keep preprocessing inside each base pipeline.

## 9. Complexity, scaling, and GPU

If one base model costs C_train to fit and C_pred per prediction, M independent members cost about O(M·C_train) training work and O(M·C_pred) prediction work; members can train in parallel. Prediction/model memory also grows with M.

For millions of rows, limit tree depth/leaves, use subsampling, parallelize carefully, and benchmark model size and inference latency. Tree ensembles are branch-heavy and often less suited to GPU than dense neural or linear algebra workloads. GPU availability depends on the specific library and tree algorithm.

In [6]:
# Reproducibility and shape checks
again = fit_bagging_regressor(X_train, y_train, n_estimators=5, max_depth=4, random_state=SEED)
again2 = fit_bagging_regressor(X_train, y_train, n_estimators=5, max_depth=4, random_state=SEED)
p1, p2 = predict_bagging_regressor(again, X_test), predict_bagging_regressor(again2, X_test)
assert np.allclose(p1, p2)
assert p1.shape == y_test.shape
try:
    fit_bagging_regressor(np.array([1,2,3]), np.array([1,2,3]), n_estimators=3)
except ValueError:
    pass
else:
    raise AssertionError("Expected bad X shape to raise")
print("All ensemble-foundation checks passed.")


All ensemble-foundation checks passed.
